учитвыая задачу мы не хотим домножать на конастану скейлер после N успешных прогонов тк мы хотим максиманльо оптимальные результаты

Поэтому хочется прогнать градиенты без amd и посмотреть на статистики, так как scaler и так может смотреть на градиенты, то статистику можно посмотреть через него

In [1]:
#!bash download_data.sh

In [1]:
from train import *
import matplotlib.pyplot as plt
scaler_test = LossScaler()

stats = dict(zip(["max", "min"], [[],[]]))

def plot_stats(optimizer):
    max_grad = -float('inf')
    min_grad = float('inf')
    for group in optimizer.param_groups:
        for p in group['params']:
            if p.grad is not None:
                grad = p.grad.data
                max_grad = max(max_grad, grad.max().item())
                min_grad = min(min_grad, grad.min().item())

    stats['max'].append(max_grad)
    stats['min'].append(min_grad)
    optimizer.step()

scaler_test.step = plot_stats

In [10]:
train(scaler_test)

plt.plot(stats['max'], label='max')
plt.plot(stats['min'], label='min')
plt.legend()
plt.show()

  0%|          | 0/40 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [6]:
# В случае статики можно взять интервал [-a, a] в котором точно будут лежать градиенты и обозначить
# scaling factor = max_value / a
scaler_static = LossScaler(scaler_type="static", interval = 0.3)
train(scaler_static)

Loss: 127584.5469 Accuracy: 98.9309: 100%|██████████| 40/40 [01:19<00:00,  1.98s/it]


In [ ]:
# В случае динамики, тк у нас всего 5 эпох и ~ 200 запусков можно накинуть ema по максимальным значениям, 
# тк максимум монотонно убывает со спайками (для данного случая), спайки мы можем пропускать, иначе без пропусков спайков у нас останется static
# scaler + спайки могут возникать изза шумных данных
scaler_static = LossScaler(scaler_type="dynamic", interval = 0.3)
train(scaler_static)

Loss: 0.5857 Accuracy: 98.7457:  28%|██▊       | 11/40 [00:24<00:46,  1.60s/it]

Inf in grad!


Loss: 0.585 Accuracy: 98.702:  30%|███       | 12/40 [00:24<00:35,  1.25s/it]  

Inf in grad!


Loss: 0.5851 Accuracy: 98.7864:  60%|██████    | 24/40 [00:47<00:19,  1.23s/it]